In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import os
import re
import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image, ImageOps
import cv2
import matplotlib.pyplot as plt
from rembg import remove
from pillow_heif import register_heif_opener
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
data_dir = "Cursive_extracted/Cursive"
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

Using Machine Learning write a program to identify handwritten letters. 

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

It was provided to me as a zip and i extracted it using 

!unzip -q Cursive-20251028T004936Z-1-001.zip -d Cursive_extracted



# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

The data contains 34 folders of letters however some of folders have only a few letters instead of all of them. Also some letters such as the letter "a" were written in multiple ways. I also noticed that the file types were different which indicates that they need to be converted prior to processing. I also saw that there were backgrounds for some of them which tells me that i need to remove it or crop the image in. The file names were different as well indicating I would need to manually label the ones that had titles unrelated to the content. 

In [2]:
def extract_label(filename):
    name = os.path.splitext(filename)[0].upper()
    if name.startswith(("IMG", "DSC", "PXL", "PHOTO", "SCREENSHOT")):
        return None
    match = re.match(r"^[A-Z]($|[_\-\s])", name)
    if match:
        return match.group(0)[0]
    return None

labels = []
for root, dirs, files in os.walk(data_dir):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.heic')):
            label = extract_label(f)
            if label:
                labels.append(label)

counts = Counter(labels)

for letter, count in sorted(counts.items()):
    print(f"{letter}: {count}")

print("\nTotal labeled images:", sum(counts.values()))


A: 34
B: 35
C: 34
D: 33
E: 34
F: 34
G: 31
H: 33
I: 27
J: 32
K: 34
L: 40
M: 36
N: 32
O: 33
P: 31
Q: 26
R: 28
S: 33
T: 27
U: 29
V: 30
W: 34
X: 32
Y: 34
Z: 32

Total labeled images: 838


# 4.Prepare the Data


Apply any data transformations and explain what and why


I added a manual labeler to handle some of the ones that had no labels in the title. I also converted the heic images to pngs. I added a preprocessing function as well. 

In [3]:
def extract_label(filename):
    name = os.path.splitext(filename.strip())[0].upper()
    if name.startswith("IMG"):
        return None
    match = re.match(r"^[A-Z]($|[_\-\s])", name)
    if match:
        return match.group(0)[0]
    return None

all_files = []
for root, _, files in os.walk(data_dir):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.heic')):
            all_files.append(os.path.join(root, f))

labels = [extract_label(os.path.basename(f)) for f in all_files]
df = pd.DataFrame({"filepath": all_files, "label": labels})
print(df["label"].value_counts(dropna=False))

unlabeled = df[df["label"].isna()]["filepath"].tolist()
print(f"Found {len(unlabeled)} unlabeled images.")

label
L    40
M    36
B    35
F    34
K    34
W    34
A    34
Y    34
E    34
C    34
O    33
S    33
H    33
D    33
X    32
J    32
Z    32
N    32
P    31
G    31
V    30
U    29
R    28
I    27
T    27
Q    26
Name: count, dtype: int64
Found 0 unlabeled images.


In [4]:
register_heif_opener()
  
converted_count = 0
for root, _, files in os.walk(data_dir):
    for f in files:
        if f.lower().endswith(".heic"):
            heic_path = os.path.join(root, f)
            png_path = os.path.splitext(heic_path)[0] + ".png"
            try:
                img = Image.open(heic_path)
                img.save(png_path, "PNG")
                converted_count += 1
                os.remove(heic_path)
            except Exception as e:
                print(f"Failed to convert {heic_path}: {e}")



In [5]:
for i, img_path in enumerate(unlabeled):
    folder = os.path.dirname(img_path)
    ext = os.path.splitext(img_path)[1]
    filename = os.path.basename(img_path)

    try:
        img = Image.open(img_path)
    except:
        continue

    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{i+1}/{len(unlabeled)} — {filename}")
    plt.show(block=False)
    plt.pause(0.1)

    try:
        label = input("Enter label (or 'quit' to stop): ").strip()
    except KeyboardInterrupt:
        plt.close()
        break

    plt.close()
    if not label:
        continue
    if label.lower() == "quit":
        break

    new_name = f"{label.upper()}{ext}"
    new_path = os.path.join(folder, new_name)
    counter = 1
    while os.path.exists(new_path):
        new_name = f"{label.upper()}_{counter}{ext}"
        new_path = os.path.join(folder, new_name)
        counter += 1

    try:
        os.rename(img_path, new_path)
        print(f"{filename} → {new_name}")
    except:
        continue

In [15]:
def has_white_background(img):
    img_rgb = img.convert('RGB')
    pixels = np.array(img_rgb)
    white_pixels = np.all(pixels > 240, axis=-1)
    white_ratio = np.sum(white_pixels) / white_pixels.size
    return white_ratio > 0.7
def preprocess_image(path, size=(224, 224), remove_bg=True):
    try:
        img = Image.open(path).convert("RGB")
        if remove_bg:
            img_np = np.array(img)
            img = Image.fromarray(remove(img_np))
            img = img.convert("RGB")
        img = ImageOps.expand(img, border=0, fill='white')
        img = img.resize(size, Image.Resampling.LANCZOS)
        img_np = np.asarray(img, dtype=np.float32) / 255.0
        img_np = (img_np - 0.5) / 0.5
        return img_np
    except Exception as e:
        print(f"Failed preprocessing {path}: {e}")
        return None


# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


In [16]:
filepaths, labels = [], []
for root, _, files in os.walk(data_dir):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            filepaths.append(os.path.join(root, f))
            labels.append(os.path.splitext(f)[0][0].upper())

encoder = LabelEncoder()
encoded_labels = encoder.fit_transform(labels)
joblib.dump(encoder, "label_encoder.pkl")

X_train_p, X_test_p, y_train, y_test = train_test_split(
    filepaths, encoded_labels, test_size=0.2, random_state=67)

In [17]:
def load_images(paths, labels):
    imgs, valid_labels = [], []
    for p, lbl in tqdm(zip(paths, labels), total=len(paths), desc="Loading images"):
        try:
            img = preprocess_image(p)
            imgs.append(img)
            valid_labels.append(lbl)
        except Exception as e:
            print(f"Skipping {p}: {e}")
    return np.array(imgs, dtype=np.float32), np.array(valid_labels)

X_train, y_train = load_images(X_train_p, y_train)
X_test, y_test   = load_images(X_test_p, y_test)

X_train_t = torch.tensor(X_train).permute(0, 3, 1, 2)
X_test_t  = torch.tensor(X_test).permute(0, 3, 1, 2)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

train_data = TensorDataset(X_train_t, y_train_t)
test_data  = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=16)

Loading images:   1%|          | 7/670 [00:11<17:43,  1.60s/it]


KeyboardInterrupt: 

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def create_cnn(num_classes):
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(128, num_classes)
    )
    
num_classes = len(encoder.classes_)
model = create_cnn(num_classes).to(device)

In [12]:
def train_model(model, loader, epochs=25, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}: loss = {running_loss/len(loader):.4f}")
    return model

model = train_model(model, train_loader, epochs=25)

Epoch 1/25:   0%|          | 0/42 [00:00<?, ?it/s]


RuntimeError: Given groups=1, weight of size [32, 3, 3, 3], expected input[16, 4, 224, 224] to have 3 channels, but got 4 channels instead

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"Accuracy: {accuracy:.3f}%")
    return accuracy
evaluate_model(model, test_loader)

# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 


# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 
